<center><h1> Map of Competitions </h1></center>

This notebook generates an interactive map of the WCA competitions of a desired competitor.
How to use this:
- download the WCA database in [here](https://www.worldcubeassociation.org/results/misc/export.html)
- change the WCAID down below
- run the notebook!

Please note that the map is centered around Europe since all my competitions are there. Points are still generated worldwide but you will have to change x and y ranges to "zoom out" the view. 

### WCAID

In [1]:
wcaid = '2014RAPO01' #change to desired WCAID

### Imports

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (12, 6)

In [3]:
import os
from bokeh.io import show, output_file, save, reset_output
from bokeh.models import GMapOptions
from bokeh.plotting import gmap
from bokeh.io import output_file
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
from bokeh.models import HoverTool
from bokeh.models import WMTSTileSource

Results:

In [4]:
Results = pd.read_csv('WCA_export_Results.tsv', sep='\t')

Competitions:

In [5]:
Competitions = pd.read_csv('WCA_export_Competitions.tsv', sep='\t')

The <i>Results</i> and <i>Competitions</i> datasets are the main ones we are going to be using. 

For every result at each competition I will need to access the information about said competition, so I merge the two datasets:

In [6]:
df = Results.merge(Competitions, how='left', left_on='competitionId', right_on='id', validate = "m:1")
df = df.drop('id', 1)

In [7]:
df = df.rename(columns = {'name':'competitionName'})

In the early stages of the WCA, some results were not registered correctly. It can happen that final rounds appear before first rounds for some competitions. To fix this, I import the <i>RoundTypes</i> table, containing an index - the <i>rank</i> - that will allow me to sort them correctly, and merge it with the dataset:

In [8]:
rounds = pd.read_csv('WCA_export_RoundTypes.tsv', sep='\t', low_memory = False)

In [9]:
df = df.merge(rounds[['id','rank']], how='left', left_on='roundTypeId', right_on='id')
df = df.drop('id',1)

The final dataset looks like this:

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2870882 entries, 0 to 2870881
Data columns (total 38 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   competitionId          object
 1   eventId                object
 2   roundTypeId            object
 3   pos                    int64 
 4   best                   int64 
 5   average                int64 
 6   personName             object
 7   personId               object
 8   personCountryId        object
 9   formatId               object
 10  value1                 int64 
 11  value2                 int64 
 12  value3                 int64 
 13  value4                 int64 
 14  value5                 int64 
 15  regionalSingleRecord   object
 16  regionalAverageRecord  object
 17  competitionName        object
 18  cityName               object
 19  countryId              object
 20  information            object
 21  year                   int64 
 22  month                  int64 
 23  day    

### List of competitions

In [11]:
comps = df[df['personId'] == wcaid].copy(deep = True)
comps['date'] = pd.to_datetime(comps[['year','month','day']])
comps = comps.drop_duplicates(subset = 'competitionId', keep = 'first')
comps = comps[~(comps['competitionId'].isin(['XA', 'XE', 'XF', 'XM', 'XN', 'XO', 'XS', 'XW']))] #(1)
comps['latitude'] = comps['latitude'] / 1000000
comps['longitude'] = comps['longitude'] / 1000000
comps = comps[['personId','personName','competitionId','date','latitude','longitude']].sort_values(by = 'date', ascending = True).reset_index(drop = True)
comps = comps.rename(columns = {'personId':'WCAID','personName':'Name','competitionId':'Competition'})
comps.index += 1

In [12]:
#remove multiple-country competitions with random coordinates 
comps = comps.drop(comps.loc[comps['Competition'].isin(['FMCEurope2015','FMCEurope2018','FMCEurope2019', 'FMC2019'])].index)

#I removed this by hand since there is these are not categorized as in (1)

In [13]:
comps

,WCAID,Name,Competition,date,latitude,longitude
1,2014RAPO01,Tommaso Raposio,MilanWinterOpen2014,2014-01-12,45.520601,9.225575
2,2014RAPO01,Tommaso Raposio,BigCubingItaly2014,2014-05-31,45.520901,9.225940
3,2014RAPO01,Tommaso Raposio,BugellaOpen2014,2014-09-06,45.549726,8.067332
4,2014RAPO01,Tommaso Raposio,PoliMiItalianOpen2014,2014-12-13,45.479632,9.226588
5,2014RAPO01,Tommaso Raposio,MilanWinterOpen2015,2015-01-04,45.520621,9.225715
6,2014RAPO01,Tommaso Raposio,DuevilleOpen2015,2015-02-22,45.634823,11.548290
8,2014RAPO01,Tommaso Raposio,TopkekOpen2015,2015-03-29,45.815304,8.812136
9,2014RAPO01,Tommaso Raposio,PoliMiOpen2015,2015-05-30,45.479632,9.226588
10,2014RAPO01,Tommaso Raposio,MantovaOpen2015,2015-10-11,45.157991,10.794668
11,2014RAPO01,Tommaso Raposio,BugellaOpen2015,2015-10-31,45.549726,8.067326


### Creating the Map

I create an empty plot centered on Europe since it's where all of my competitions took place

In [14]:
# web mercator coordinates
EU = x_range,y_range = ((-1500000,4500000), (4500000,9000000))

#change these to zoom out

This allows bokeh to request for specific tiles once the plot is created

In [15]:
url = 'http://a.basemaps.cartocdn.com/rastertiles/voyager/{Z}/{X}/{Y}.png'
attribution = "Tiles by Carto, under CC BY 3.0. Data by OSM, under ODbL"

### Filling the map

Bokeh works with the Web Mercator Format. This function converts latitude and longitude.

In [16]:
def wgs84_to_web_mercator(df, lon="longitude", lat="latitude"):
    """Converts decimal longitude/latitude to Web Mercator format"""
    k = 6378137
    df["mercator_x"] = df[lon] * (k * np.pi/180.0)
    df["mercator_y"] = np.log(np.tan((90 + df[lat]) * np.pi/360.0)) * k
    return df

In [17]:
comps = wgs84_to_web_mercator(comps) #conversion

In [18]:
output_file(filename = f"CompMap{wcaid}.html")

fonte = ColumnDataSource(comps.to_dict(orient='list'))

#Hovertool
hover = HoverTool(tooltips=[("number", "$index"),("Competition", "@Competition")])


#create figure
p = figure(tools=['pan, wheel_zoom, reset',hover], title = 'Interactive map of '+str(wcaid)+"'s competitions", x_range=x_range, y_range=y_range, 
           x_axis_type="mercator", y_axis_type="mercator")

#add tiles
p.add_tile(WMTSTileSource(url=url, attribution=attribution))

#add glyphs
p.circle(source = fonte, x='mercator_x', y='mercator_y', fill_color='blue', size=10)

#add labels
#labels = LabelSet(x='mercator_x', y='mercator_y', text='Competition', level='glyph',
                  #x_offset=5, y_offset=5, source=fonte, render_mode='canvas')


#p.add_layout(labels)

save(p)

reset_output()

### Graph

In [19]:
output_file(filename = f"CompMapGraph{wcaid}.html")

fonte2 = ColumnDataSource(comps.to_dict(orient='list'))

hover2 = HoverTool(tooltips=[("number", "$index"),("Competition", "@Competition")])

q = figure(tools=['pan, wheel_zoom, reset',hover2], title = 'Interactive map of '+str(wcaid)+"'s competitions", x_range=x_range, y_range=y_range, 
           x_axis_type="mercator", y_axis_type="mercator")

#add tiles
q.add_tile(WMTSTileSource(url=url, attribution=attribution))

#add glyphs
q.circle(source = fonte2, x='mercator_x', y='mercator_y', fill_color='blue', size=10)
q.line(source = fonte2, x='mercator_x', y='mercator_y', line_width=1, line_color = 'orange', alpha = 0.6)
       
save(q)

reset_output()